# MCP 模型上下文协议

## MCP 是什么

MCP 是 Model Context Protocol 的缩写，是连接大模型与外部资源/工具的标准化接口服务。 说的通俗一些mcp就是标准化的function call，只不过这个function call是用于大模型与外部资源/工具之间的交互。 我们知道大模型本身只是对数据进行计算和处理，本身不具备获取外部资源和工具的能力，而mcp为大模型提供了具备调用外部资源和工具的能力，并且mcp服务的诞生可以让大模型自行调用这些资源和工具。 比如大模型没有能力调用我们本地数据库数据，然后使用特定结构的SQL查询获取数据再生成图表，那么我们现在就可以使用mcp来实现这个功能。一个 mcp 负责通过SQL查询获取数据，然后另一个 mcp 负责生成图表,比如图片格式或者HTML格式。

## 我们将构建什么
许多大语言模型目前没有能力获取天气预报和恶劣天气警报。让我们使用 MCP 来解决这个问题！

我们将构建一个服务器，提供两个工具：get-alerts 和 get-forecast。然后我们将服务器连接到 MCP 主机（在本例中是 Claude for Desktop）：

服务器可以连接到任何客户端。我们在这里选择 Claude for Desktop 是为了简单起见，但我们也有关于构建自己的客户端的指南以及此处的其他客户端列表。

由于服务器是本地运行的，MCP 目前只支持桌面主机。远程主机正在积极开发中。


## Core MCP Concepts - MCP 核心概念

MCP 服务器可以提供三种主要类型的功能：

- `Resources`（资源）：客户端可以读取的类似文件的数据（如 API 响应或文件内容） 也就是说我们自己开发好的后端API或者第三方的API接口或者是文件等，可以通过 Resource 来进行读取的。区别在于传统API调用是我们或程序主动请求和处理数据的过程，但是Resource方式则是将数据源注册为标准化URI资源，允许大模型通过统一接口直接访问，无需关心底层实现细节。也就是说我们有一个URL，这个URL可以直接接入数据库某个表的数据，然后大模型就可以直接通过这个URL来获取这个表的数据。并且不会像使用 tools 那样需要我们进行确认。这样的做法丰富了大模型的上下文信息。标准规范是 Resource 只对数据进行读取，不能进行写入。
- `Tools`（工具）：LLM 可以调用的函数（需用户批准） 也就是说我们开发的 MCP 服务可以提供很多函数，这些函数可以让大模型自行调用，比如我们开发了一个获取天气预报的函数，那么大模型就可以自己调用这个函数来获取天气预报信息。那么类比一下就好比我们传统开发后端API一样，我们自己开发了一个获取天气预报的API，然后我们自己调用这个API来获取天气预报信息。只不过这个流程是把我们自己替换成大模型而已。
- `Prompts`（提示）：帮助用户完成特定任务的预写模板 其实就是为Tools提供一个提示词，让大模型可以根据提示词模板进行回答。就比如说我通过天气预报函数过去今天的天气的同时我还想让大模型给我出门穿衣的建议，那么我就可以为这个天气函数提供一个提示词，让大模型可以根据提示词模板进行回答。




## 前置知识要求
本快速入门假设您熟悉：
   
Python    
像 Claude 这样的大语言模型    
系统要求     
安装 Python 3.10 或更高版本。     
您必须使用 Python MCP SDK 1.2.0 或更高版本。     

## 设置您的环境
首先，让我们安装 uv 并设置我们的 Python 项目和环境： uv 是一个 Python 依赖管理工具，类似我们开发node项目需要使用 npm 或 npx 依赖管理工具一样。 那么 Python 的依赖管理工具我们常用的还有 pip 和 venv，比如上期视频我就使用 venv 创建了一个python的虚拟环境去启动一个mcp服务。 只不过 uv 比 pip 和 venv 更快，但实际如何其实我也没多大感受，只不过既然官方文档提供使用uv的方式，那我们就按照官方文档的内容的来做。 至于 uv 与 pip 和 venv 的特点大家可以使用 PPL MCP 服务在cursor中提问获取最新的信息进行比对即可，PPL MCP 服务是我第一个视频讲解到的。

```shell
curl -LsSf https://astral.sh/uv/install.sh | sh
```


## 初始化 FastMCP Server
MCP Python SDK 现在提供了全新的 FastMCP 类，它通过利用 Python 的类型注解（Type Hints）和文档字符串（Docstrings）特性，能够自动生成工具定义。这种方式让开发者可以更加便捷地创建和管理 MCP 的 Tool、Resource 以及 Prompt 等功能组件。

以下代码创建一个名为 mcp 的 FastMCP 对象。

```python
from mcp.server.fastmcp import FastMCP

MCP_SERVER_NAME = "elasticsearch-mcp-server"
mcp = FastMCP(MCP_SERVER_NAME)
```

## 添加 Tool

Tool 定义了允许 LLM 可以调用 MCP Server 执行的操作，除了查询以外，还可以执行写入操作。接下来定义了两个 Tool：

- `list_indices`: 列出所有可用的索引。

- `get_index`: 获取指定索引的详细信息。

使用 @mcp.tool() 装饰器将这两个函数标记为 MCP 的 Tool。


```python
@mcp.tool()
def list_indices() -> List[str]:
    """列出所有 Elasticsearch 索引"""
    return [index["index"] for index in es.cat.indices(format="json")]

@mcp.tool()
def get_index(index: str) -> dict:
    """获取特定 Elasticsearch 索引的详细信息"""
    return es.indices.get(index=index)

```



## 添加 Resource
Resource 定义了 LLM 可以访问只读的数据源，可以用于为 LLM 提供上下文内容。在这个示例中，我们定义了两个资源：

- es://logs：允许 LLM 访问 Elasticsearch 容器的日志信息，通过 Docker 命令获取日志内容。
- file://docker-compose.yaml：允许 LLM 访问项目的 docker-compose.yaml 文件内容。


```python
@mcp.resource("es://logs")
def get_logs() -> str:
    """Get Elasticsearch container logs"""
    result = subprocess.run(["docker", "logs", "elasticsearch-mcp-server-example-es01-1"], capture_output=True, text=True, check=True)
    return result.stdout

@mcp.resource("file://docker-compose.yaml")
def get_file() -> str:
    """Return the contents of docker-compose.yaml file"""
    with open("docker-compose.yaml", "r") as f:
        return f.read()

```

使用 `@mcp.resource()` 装饰器将这些函数标记为 MCP 的 Resource，装饰器参数指定了 Resource 的 URI。

### Resource 建议遵循以下格式的 URI 标识：
```bash
[protocol]://[host]/[path]
```

### 添加 Prompt

Prompt 用于定义可重用的提示模板，帮助用户更好地引导 LLM 以标准化的方式完成任务。在这个示例中，我们定义了一个名为 `es_prompt` 的提示模板，引导 LLM 从多个维度（如索引设置、搜索优化、数据建模和扩展性等）对索引进行分析。

```python
@mcp.prompt()
def es_prompt(index: str) -> str:
    """Create a prompt for index analysis"""
    return f"""You are an elite Elasticsearch expert with deep knowledge of search engine architecture, data indexing strategies, and performance optimization. Please analyze the index '{index}' considering:
- Index settings and mappings
- Search optimization opportunities
- Data modeling improvements
- Potential scaling considerations

```



## 使用 MCP Inspector 调试 MCP Server
MCP Inspector 是一个交互式的开发者工具，专门用于测试和调试 MCP 服务器。它提供了一个图形化界面，让开发者能够直观地检查和验证 MCP 服务器的功能。
执行以下命令可以启动 MCP Inspector：

```python
mcp dev server.py 
```

启动成功后，浏览数输入 http://localhost:5173 打开 MCP Inspector 界面。点击 Connect 连接 MCP Server。